In [1]:
# 00_understanding_states.py
"""
FlagQuantum 入门教程 - 第零课：理解状态张量的结构
目标：彻底理解 FlagQuantum 如何表示量子状态
"""

import torch

import flagquantum as fq


def tutorial_00_basic_tensor_structure():
    """理解基础张量结构"""
    print("=" * 70)
    print("第 0.0 课：状态张量的基本结构")
    print("=" * 70)

    # 创建不同规模的设备
    configs = [
        (1, "单量子比特"),
        (2, "双量子比特"),
        (3, "三量子比特"),
    ]

    for n_wires, name in configs:
        qdev = fq.DistributedQuantumDevice(n_wires=n_wires, bsz=1, device="cpu")
        print(f"\n{name}: n_wires={n_wires}")
        print(f"  状态张量形状: {qdev.states.shape}")
        print(f"  解释: {qdev.states.shape} = [batch_size, {', '.join([f'q{i}' for i in range(n_wires)])}, real/imag]")


In [2]:
tutorial_00_basic_tensor_structure()

第 0.0 课：状态张量的基本结构

单量子比特: n_wires=1
  状态张量形状: torch.Size([1, 2, 2])
  解释: torch.Size([1, 2, 2]) = [batch_size, q0, real/imag]

双量子比特: n_wires=2
  状态张量形状: torch.Size([1, 2, 2, 2])
  解释: torch.Size([1, 2, 2, 2]) = [batch_size, q0, q1, real/imag]

三量子比特: n_wires=3
  状态张量形状: torch.Size([1, 2, 2, 2, 2])
  解释: torch.Size([1, 2, 2, 2, 2]) = [batch_size, q0, q1, q2, real/imag]


In [3]:

"""
FlagQuantum 端序速查卡
===========================================
端序类型: Big-endian ✅
形状格式: [batch, q0, q1, ..., q_{N-1}, 2]
比特顺序: q0 = MSB (最高有效位), q_{N-1} = LSB

基态示例 (n=3):
  |000⟩ → [batch, 0,0,0, :]
  |001⟩ → [batch, 0,0,1, :]   (q2=1)
  |010⟩ → [batch, 0,1,0, :]   (q1=1)
  |100⟩ → [batch, 1,0,0, :]   (q0=1)
"""

def test_endianness():
    """通过手动计算概率来验证 FlagQuantum 的端序"""
    print("\n" + "=" * 70)
    print("第 0.1 课：确定 Qubit 排序（Big-endian vs Little-endian）")
    print("=" * 70)
    # 1. 创建 2 量子比特设备，初始状态为 |00>
    qdev = fq.DistributedQuantumDevice(n_wires=2, bsz=1, device="cpu")

    # 辅助函数：从 states 张量计算概率分布
    def get_probs(device):
        # 假设 states 形状为 [batch, q0, q1, ..., 2]
        real = device.states[..., 0]
        imag = device.states[..., 1]
        state_vector = torch.complex(real, imag)
        # 计算模平方得到概率，并展平为一维数组以便查看
        probs = (state_vector.abs()) ** 2
        return probs.flatten()

    print("=== FlagQuantum 端序测试 ===")
    print(f"states 原始形状: {qdev.states.shape}")

    # 2. 查看初始状态概率
    probs_initial = get_probs(qdev)
    print(f"\n初始状态概率 (应为 |00> = 1.0):\n  {probs_initial}")

    # 3. 对第一个量子比特 (wires=0) 施加 X 门
    qdev.x(wires=0)
    probs_after_x = get_probs(qdev)
    print(f"\n对 q0 施加 X 门后概率:\n  {probs_after_x}")

    # 4. 判断端序
    # Little-endian 期望：索引 1 (二进制 01) 概率为 1
    # Big-endian 期望：索引 2 (二进制 10) 概率为 1
    if probs_after_x[1] > 0.99:
        print("\n✅ 结论: Little-endian (小端序)")
        print("   解释: q0 是 LSB，|00> -> X(q0) -> |01>")
    elif probs_after_x[2] > 0.99:
        print("\n✅ 结论: Big-endian (大端序)")
        print("   解释: q0 是 MSB，|00> -> X(q0) -> |10>")
    else:
        print("\n⚠️ 无法判断，请检查门操作是否生效或比特数是否正确")

    # 5. 可选：测试 CNOT 进一步验证
    print("\n--- 进阶验证：CNOT 门 ---")
    # 重置设备或创建新的 (这里简单新建一个)
    qdev2 = fq.DistributedQuantumDevice(n_wires=2, bsz=1, device="cpu")
    # 制备 |01> 态 (在 Little-endian 中为 q0=1)
    qdev2.x(wires=0)
    print(f"制备 |01> 态概率: {get_probs(qdev2)}")

    # CNOT: control=q0, target=q1
    qdev2.cx(wires=[0, 1])
    probs_cnot = get_probs(qdev2)
    print(f"CNOT(q0->q1) 作用后: {probs_cnot}")

    if probs_cnot[3] > 0.99:  # 索引 3 对应 |11> (二进制 11)
        print("CNOT 验证通过: 控制位和目标的交互逻辑符合预期")

if __name__ == "__main__":
    test_endianness()


第 0.1 课：确定 Qubit 排序（Big-endian vs Little-endian）
=== FlagQuantum 端序测试 ===
states 原始形状: torch.Size([1, 2, 2, 2])

初始状态概率 (应为 |00> = 1.0):
  tensor([1., 0., 0., 0.])

对 q0 施加 X 门后概率:
  tensor([0., 0., 1., 0.])

✅ 结论: Big-endian (大端序)
   解释: q0 是 MSB，|00> -> X(q0) -> |10>

--- 进阶验证：CNOT 门 ---
制备 |01> 态概率: tensor([0., 0., 1., 0.])
CNOT(q0->q1) 作用后: tensor([0., 0., 0., 1.])
CNOT 验证通过: 控制位和目标的交互逻辑符合预期


In [4]:
def tutorial_02_read_single_qubit_state():
    """读取和理解状态向量"""
    print("\n" + "=" * 70)
    print("第 0.2 课：读取和理解状态向量")
    print("=" * 70)

    qdev = fq.DistributedQuantumDevice(n_wires=2, bsz=1, device="cpu")

    print("\n1. 基础状态：|00⟩")
    states = torch.view_as_complex(qdev.states)
    print(f"   张量形状: {states.shape}")
    print(f"   状态: {states}")
    print(f"   展开: {states.flatten()}")
    print("   含义: states[0,0] = 1.0 → 100% 概率在 |00⟩")

    print("\n2. 应用 X 门在 qubit 0：|10⟩")
    qdev.reset_states()
    fq.X(wires=[0])(qdev)
    states = torch.view_as_complex(qdev.states)
    print(f"   状态: {states.flatten()}")
    nonzero = torch.where(states.abs() > 0.5)
    print(f"   非零位置: {nonzero}")

    print("\n3. 应用 X 门在 qubit 1：|01⟩")
    qdev.reset_states()
    fq.X(wires=[1])(qdev)
    states = torch.view_as_complex(qdev.states)
    print(f"   状态: {states.flatten()}")
    nonzero = torch.where(states.abs() > 0.5)
    print(f"   非零位置: {nonzero}")

    print("\n4. 应用 X 门在两个 qubit：|11⟩")
    qdev.reset_states()
    fq.X(wires=[0])(qdev)
    fq.X(wires=[1])(qdev)
    states = torch.view_as_complex(qdev.states)
    print(f"   状态: {states.flatten()}")

In [5]:
tutorial_02_read_single_qubit_state()


第 0.2 课：读取和理解状态向量

1. 基础状态：|00⟩
   张量形状: torch.Size([1, 2, 2])
   状态: tensor([[[1.+0.j, 0.+0.j],
         [0.+0.j, 0.+0.j]]])
   展开: tensor([1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j])
   含义: states[0,0] = 1.0 → 100% 概率在 |00⟩

2. 应用 X 门在 qubit 0：|10⟩
   状态: tensor([0.+0.j, 0.+0.j, 1.+0.j, 0.+0.j])
   非零位置: (tensor([0]), tensor([1]), tensor([0]))

3. 应用 X 门在 qubit 1：|01⟩
   状态: tensor([0.+0.j, 1.+0.j, 0.+0.j, 0.+0.j])
   非零位置: (tensor([0]), tensor([0]), tensor([1]))

4. 应用 X 门在两个 qubit：|11⟩
   状态: tensor([0.+0.j, 0.+0.j, 0.+0.j, 1.+0.j])


In [6]:
def tutorial_03_amplitude_to_probability():
    """从振幅到概率"""
    print("\n" + "=" * 70)
    print("第 0.3 课：从振幅到概率")
    print("=" * 70)

    qdev = fq.DistributedQuantumDevice(n_wires=2, bsz=1, device="cpu")

    # 创建叠加态
    fq.H(wires=[0])(qdev)
    fq.H(wires=[1])(qdev)

    states = torch.view_as_complex(qdev.states)
    print(f"叠加态振幅: {states.flatten()}")
    print(f"所有振幅都是复数: {states.dtype}")

    # 计算概率
    probs = torch.abs(states) ** 2
    print(f"\n概率分布: {probs.flatten()}")
    print(f"概率和: {probs.sum().item():.1f} (应为 1.0)")

    # 验证归一化
    print("\n验证归一化:")
    for i in range(4):
        binary = format(i, '02b')
        prob = probs.flatten()[i].item()
        amp = states.flatten()[i]
        print(f"  |{binary}⟩: 振幅={amp:.3f}, 概率={prob:.3f}")

In [7]:
tutorial_03_amplitude_to_probability()


第 0.3 课：从振幅到概率
叠加态振幅: tensor([0.5000+0.j, 0.5000+0.j, 0.5000+0.j, 0.5000+0.j])
所有振幅都是复数: torch.complex64

概率分布: tensor([0.2500, 0.2500, 0.2500, 0.2500])
概率和: 1.0 (应为 1.0)

验证归一化:
  |00⟩: 振幅=0.500+0.000j, 概率=0.250
  |01⟩: 振幅=0.500+0.000j, 概率=0.250
  |10⟩: 振幅=0.500+0.000j, 概率=0.250
  |11⟩: 振幅=0.500+0.000j, 概率=0.250


In [8]:
def tutorial_04_batch_processing():
    """批处理状态"""
    print("\n" + "=" * 70)
    print("第 0.4 课：批处理状态")
    print("=" * 70)

    batch_size = 3
    n_wires = 2
    qdev = fq.DistributedQuantumDevice(n_wires=n_wires, bsz=batch_size, device="cpu")

    print(f"批次大小: {batch_size}")
    print(f"状态张量形状: {qdev._states.shape}")
    print(f"解释: [{batch_size}, {n_wires}个qubit维度, real/imag]")

    # 每个批次独立处理
    print("\n为每个批次设置不同的初始状态:")
    for i in range(batch_size):
        qdev.reset_states()  # 重置所有批次
        # 这里只能整体设置，不能单独设置每个批次
        # 需要通过参数化门来实现批次独立性

    # 使用批处理参数
    thetas = torch.tensor([[0.5], [1.0], [1.5]])  # 每个批次不同角度
    print(f"批次参数: {thetas.squeeze()}")

    fq.RY(wires=[0])(qdev, params=thetas)

    states = torch.view_as_complex(qdev.states)
    print("\n各批次状态:")
    for i in range(batch_size):
        probs = torch.abs(states[i]) ** 2
        print(f"  批次 {i}: 概率 |0⟩={probs[0,0].item():.3f}, |1⟩={probs[1,0].item():.3f}")

In [9]:
tutorial_04_batch_processing()


第 0.4 课：批处理状态
批次大小: 3
状态张量形状: torch.Size([3, 2, 2, 2])
解释: [3, 2个qubit维度, real/imag]

为每个批次设置不同的初始状态:
批次参数: tensor([0.5000, 1.0000, 1.5000])

各批次状态:
  批次 0: 概率 |0⟩=0.939, |1⟩=0.061
  批次 1: 概率 |0⟩=0.770, |1⟩=0.230
  批次 2: 概率 |0⟩=0.535, |1⟩=0.465


In [10]:
def tutorial_05_complex_numbers():
    """理解复数振幅"""
    print("\n" + "=" * 70)
    print("第 0.5 课：理解复数振幅")
    print("=" * 70)

    qdev = fq.DistributedQuantumDevice(n_wires=1, bsz=1, device="cpu")

    # 只有实数的情况
    print("1. 实数振幅（H 门）:")
    fq.H(wires=[0])(qdev)
    states = torch.view_as_complex(qdev.states)
    print(f"   {states.flatten()}")
    print("   虚部全是 0")

    # 复数出现的情况
    print("\n2. 复数振幅（Phase 门）:")
    qdev.reset_states()
    fq.H(wires=[0])(qdev)  # 先创建叠加态
    fq.PHASE(wires=[0])(qdev, params=torch.tensor([0.8]))  # 添加相位
    states = torch.view_as_complex(qdev.states)
    print(f"   {states.flatten()}")
    print(f"   |0⟩: {states[0][0].item():.3f}")
    print(f"   |1⟩: {states[0][1].item():.3f}")
    print(f"   |1⟩ 的相位: {torch.atan2(states[0][1].imag, states[0][1].real).item():.3f} rad")

    # 相位的作用
    print("\n3. 相位的物理意义:")
    print("   量子态的全局相位不可观测，但相对相位可以")
    qdev.reset_states()
    fq.H(wires=[0])(qdev)
    fq.PHASE(wires=[0])(qdev, params=torch.tensor([torch.pi]))  # 添加 π 相位
    states = torch.view_as_complex(qdev.states)
    print(f"   添加 π 相位后: {states.flatten()}")
    print("   概率不变，但相对相位改变了")

In [11]:
tutorial_05_complex_numbers()


第 0.5 课：理解复数振幅
1. 实数振幅（H 门）:
   tensor([0.7071+0.j, 0.7071+0.j])
   虚部全是 0

2. 复数振幅（Phase 门）:
   tensor([0.7071+0.0000j, 0.4926+0.5072j])
   |0⟩: 0.707+0.000j
   |1⟩: 0.493+0.507j
   |1⟩ 的相位: 0.800 rad

3. 相位的物理意义:
   量子态的全局相位不可观测，但相对相位可以
   添加 π 相位后: tensor([ 0.7071+0.0000e+00j, -0.7071-6.1817e-08j])
   概率不变，但相对相位改变了
